# Notebook 8 — Comparaison Multi-Modèles (Dossiers Séparés)
**ISIC Project | U-Net vs SegNet vs DeepLabV3**

Chaque modèle a son **propre dossier** sur Google Drive :

```
ISIC_Project/outputs/
├── unet/
│   ├── best_unet.pth            ← meilleur modèle
│   ├── checkpoint_unet.pth      ← reprise automatique
│   ├── training_curves_unet.png
│   └── metrics_unet.json
├── segnet/
│   ├── best_segnet.pth
│   ├── checkpoint_segnet.pth
│   ├── training_curves_segnet.png
│   └── metrics_segnet.json
├── deeplabv3/
│   ├── best_deeplabv3.pth
│   ├── checkpoint_deeplabv3.pth
│   ├── training_curves_deeplabv3.png
│   └── metrics_deeplabv3.json
└── comparaison/
    ├── comparaison_complete.png
    ├── comparaison_visuelle.png
    └── resultats_finaux.json
```

## Étape 1 — Setup & Création des dossiers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_PATH = '/content/drive/MyDrive/ISIC_Project'
IMAGES_PATH  = os.path.join(PROJECT_PATH, 'data', 'Images')
MASQUES_PATH = os.path.join(PROJECT_PATH, 'data', 'Masques')
OUTPUTS_PATH = os.path.join(PROJECT_PATH, 'outputs')

# ── Dossier par modèle ────────────────────────────────────────────────────────
DIRS = {
    'unet'      : os.path.join(OUTPUTS_PATH, 'unet'),
    'segnet'    : os.path.join(OUTPUTS_PATH, 'segnet'),
    'deeplabv3' : os.path.join(OUTPUTS_PATH, 'deeplabv3'),
    'comparaison': os.path.join(OUTPUTS_PATH, 'comparaison'),
}
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

sys.path.append(os.path.join(PROJECT_PATH, 'src'))

print('OK Structure de dossiers créée :')
for name, path in DIRS.items():
    print(f'   outputs/{name}/')

## Étape 2 — Installations & Imports

In [ ]:
!pip install -q albumentations opencv-python-headless scikit-learn seaborn torchvision

import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models.segmentation as seg_models
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm
import json, time, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'   GPU  : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## Étape 3 — Dataset & DataLoaders

In [ ]:
IMG_SIZE   = 256
BATCH_SIZE = 8
EPOCHS     = 30
THRESHOLD  = 0.5

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3), A.ColorJitter(p=0.4), A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']; mask = out['mask'].unsqueeze(0)
        return img.float(), mask.float()

IMAGE_EXTENSIONS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
all_images = sorted([p for p in Path(IMAGES_PATH).glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS])
all_masks  = sorted([p for p in Path(MASQUES_PATH).glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS])
assert len(all_images) == len(all_masks)

train_imgs, temp_imgs, train_msks, temp_msks = train_test_split(
    all_images, all_masks, test_size=0.3, random_state=42)
val_imgs, test_imgs, val_msks, test_msks = train_test_split(
    temp_imgs, temp_msks, test_size=0.5, random_state=42)

train_loader = DataLoader(ISICDataset(train_imgs, train_msks, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ISICDataset(val_imgs,   val_msks,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ISICDataset(test_imgs,  test_msks,  val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'OK Dataset : {len(all_images)} images')
print(f'   Train:{len(train_imgs)} | Val:{len(val_imgs)} | Test:{len(test_imgs)}')

## Étape 4 — Définition des 3 architectures

In [ ]:
# ══════════════════════════════════════════════════════
# U-NET
# ══════════════════════════════════════════════════════
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    """Encodeur-Décodeur avec skip connections par concaténation."""
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        self.pool = nn.MaxPool2d(2,2)
        for f in features:
            self.downs.append(DoubleConv(in_channels, f)); in_channels = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_channels, 1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); skip = skips[i//2]
            if x.shape != skip.shape: x = F.interpolate(x, size=skip.shape[2:])
            x = self.ups[i+1](torch.cat([skip, x], dim=1))
        return self.final(x)

# ══════════════════════════════════════════════════════
# SEGNET
# ══════════════════════════════════════════════════════
class SegNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n=2):
        super().__init__()
        layers = []
        for i in range(n):
            layers += [nn.Conv2d(in_ch if i==0 else out_ch, out_ch, 3, padding=1),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class SegNet(nn.Module):
    """Encodeur VGG-like + décodeur par max-unpooling avec indices."""
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = SegNetBlock(in_channels,64,2); self.enc2 = SegNetBlock(64,128,2)
        self.enc3 = SegNetBlock(128,256,3);         self.enc4 = SegNetBlock(256,512,3)
        self.enc5 = SegNetBlock(512,512,3)
        self.dec5 = SegNetBlock(512,512,3); self.dec4 = SegNetBlock(512,256,3)
        self.dec3 = SegNetBlock(256,128,3); self.dec2 = SegNetBlock(128,64,2)
        self.dec1 = SegNetBlock(64,64,2)
        self.pool   = nn.MaxPool2d(2,2,return_indices=True)
        self.unpool = nn.MaxUnpool2d(2,2)
        self.final  = nn.Conv2d(64, out_channels, 1)
    def forward(self, x):
        x1=self.enc1(x); x,i1=self.pool(x1)
        x2=self.enc2(x); x,i2=self.pool(x2)
        x3=self.enc3(x); x,i3=self.pool(x3)
        x4=self.enc4(x); x,i4=self.pool(x4)
        x5=self.enc5(x); x,i5=self.pool(x5)
        x=self.unpool(x,i5,output_size=x5.size()); x=self.dec5(x)
        x=self.unpool(x,i4,output_size=x4.size()); x=self.dec4(x)
        x=self.unpool(x,i3,output_size=x3.size()); x=self.dec3(x)
        x=self.unpool(x,i2,output_size=x2.size()); x=self.dec2(x)
        x=self.unpool(x,i1,output_size=x1.size()); x=self.dec1(x)
        return self.final(x)

# ══════════════════════════════════════════════════════
# DEEPLABV3
# ══════════════════════════════════════════════════════
class DeepLabV3Wrapper(nn.Module):
    """ResNet50 pré-entraîné + ASPP, tête remplacée pour segmentation binaire."""
    def __init__(self, out_channels=1, pretrained=True):
        super().__init__()
        weights = seg_models.DeepLabV3_ResNet50_Weights.DEFAULT if pretrained else None
        self.model = seg_models.deeplabv3_resnet50(weights=weights, aux_loss=False)
        self.model.classifier[4] = nn.Conv2d(256, out_channels, kernel_size=1)
    def forward(self, x):
        out = self.model(x)['out']
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return out

# ── Instanciation ─────────────────────────────────────
unet    = UNet().to(device)
segnet  = SegNet().to(device)
deeplab = DeepLabV3Wrapper(pretrained=True).to(device)

print(f'U-Net     : {sum(p.numel() for p in unet.parameters()   if p.requires_grad):>12,} paramètres')
print(f'SegNet    : {sum(p.numel() for p in segnet.parameters()  if p.requires_grad):>12,} paramètres')
print(f'DeepLabV3 : {sum(p.numel() for p in deeplab.parameters() if p.requires_grad):>12,} paramètres')

# Test forward
with torch.no_grad():
    d = torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(device)
    print(f'\nOK U-Net     output : {unet(d).shape}')
    print(f'OK SegNet    output : {segnet(d).shape}')
    print(f'OK DeepLabV3 output : {deeplab(d).shape}')

## Étape 5 — Fonctions communes

In [ ]:
# ── Perte ─────────────────────────────────────────────────────────────────────
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__(); self.bce = nn.BCEWithLogitsLoss()
    def dice_loss(self, pred, target, smooth=1e-6):
        pred  = torch.sigmoid(pred)
        inter = (pred * target).sum(dim=(2,3))
        union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
        return 1 - ((2*inter+smooth)/(union+smooth)).mean()
    def forward(self, pred, target):
        return 0.5*self.bce(pred,target) + 0.5*self.dice_loss(pred,target)

criterion = DiceBCELoss()

# ── Métriques ──────────────────────────────────────────────────────────────────
def compute_metrics(pred_logit, target, threshold=THRESHOLD, smooth=1e-6):
    pred_bin = (torch.sigmoid(pred_logit) > threshold).float()
    tp = (pred_bin * target).sum(dim=(2,3))
    tn = ((1-pred_bin)*(1-target)).sum(dim=(2,3))
    fp = (pred_bin*(1-target)).sum(dim=(2,3))
    fn = ((1-pred_bin)*target).sum(dim=(2,3))
    total = tp+tn+fp+fn
    return {
        'accuracy'  : ((tp+tn)/(total+smooth)).mean().item(),
        'dice'      : ((2*tp+smooth)/(2*tp+fp+fn+smooth)).mean().item(),
        'iou'       : ((tp+smooth)/(tp+fp+fn+smooth)).mean().item(),
        'precision' : ((tp+smooth)/(tp+fp+smooth)).mean().item(),
        'recall'    : ((tp+smooth)/(tp+fn+smooth)).mean().item(),
    }

# ── Checkpoint ─────────────────────────────────────────────────────────────────
def save_ckpt(epoch, model, opt, sch, history, best_dice, path):
    torch.save({'epoch':epoch,'model_state':model.state_dict(),
                'optim_state':opt.state_dict(),'sched_state':sch.state_dict(),
                'history':history,'best_dice':best_dice}, path)

def load_ckpt(path, model, opt, sch):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    opt.load_state_dict(ckpt['optim_state'])
    sch.load_state_dict(ckpt['sched_state'])
    return ckpt['epoch'], ckpt['history'], ckpt['best_dice']

print('OK DiceBCELoss, compute_metrics, save/load_ckpt prêts !')

## Étape 6 — Fonction d'entraînement universelle

In [ ]:
def train_model(model, model_name, lr=1e-4, epochs=EPOCHS):
    """
    Entraîne un modèle et sauvegarde TOUT dans son propre dossier.

    Dossier : outputs/<model_name>/
    Fichiers créés :
      - checkpoint_<model_name>.pth  → reprise automatique si coupure
      - best_<model_name>.pth        → meilleur modèle (Dice Val)
      - training_curves_<model_name>.png
      - metrics_<model_name>.json
    """
    model_dir   = DIRS[model_name]
    ckpt_path   = os.path.join(model_dir, f'checkpoint_{model_name}.pth')
    best_path   = os.path.join(model_dir, f'best_{model_name}.pth')
    curves_path = os.path.join(model_dir, f'training_curves_{model_name}.png')
    metrics_path= os.path.join(model_dir, f'metrics_{model_name}.json')

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=False)

    # ── Reprise automatique ───────────────────────────────────────────────────
    if os.path.exists(ckpt_path):
        start_epoch, history, best_dice = load_ckpt(ckpt_path, model, optimizer, scheduler)
        start_epoch += 1
        print(f'REFRESH [{model_name}] Reprise epoch {start_epoch}/{epochs} '
              f'(best Dice: {best_dice:.4f})')
    else:
        start_epoch = 1; best_dice = 0.0
        history = {k:[] for k in ['train_loss','val_loss','train_dice','val_dice',
                                   'train_acc','val_acc','train_iou','val_iou',
                                   'train_prec','val_prec','train_rec','val_rec']}
        print(f'NEW [{model_name}] Démarrage — {epochs} epochs | lr={lr}')
        print(f'   Dossier : outputs/{model_name}/')

    t0 = time.time()

    for epoch in range(start_epoch, epochs+1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        tr = {k:0.0 for k in ['loss','dice','acc','iou','prec','rec']}
        for imgs, masks in tqdm(train_loader,
                                 desc=f'[{model_name}] {epoch}/{epochs} Train',
                                 leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss  = criterion(preds, masks)
            loss.backward(); optimizer.step()
            m = compute_metrics(preds, masks)
            tr['loss']+=loss.item(); tr['dice']+=m['dice']; tr['acc']+=m['accuracy']
            tr['iou'] +=m['iou'];   tr['prec']+=m['precision']; tr['rec']+=m['recall']
        for k in tr: tr[k] /= len(train_loader)

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        vl = {k:0.0 for k in ['loss','dice','acc','iou','prec','rec']}
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs)
                m = compute_metrics(preds, masks)
                vl['loss']+=criterion(preds,masks).item(); vl['dice']+=m['dice']
                vl['acc'] +=m['accuracy'];  vl['iou'] +=m['iou']
                vl['prec']+=m['precision']; vl['rec'] +=m['recall']
        for k in vl: vl[k] /= len(val_loader)

        scheduler.step(vl['loss'])

        # ── Historique ────────────────────────────────────────────────────────
        for k,v in [('train_loss',tr['loss']),('val_loss',vl['loss']),
                    ('train_dice',tr['dice']),('val_dice',vl['dice']),
                    ('train_acc', tr['acc']), ('val_acc', vl['acc']),
                    ('train_iou', tr['iou']), ('val_iou', vl['iou']),
                    ('train_prec',tr['prec']),('val_prec',vl['prec']),
                    ('train_rec', tr['rec']), ('val_rec', vl['rec'])]:
            history[k].append(v)

        # ── Checkpoint toutes les epochs → dossier du modèle ──────────────────
        save_ckpt(epoch, model, optimizer, scheduler, history, best_dice, ckpt_path)

        # ── Meilleur modèle → dossier du modèle ───────────────────────────────
        tag = ''
        if vl['dice'] > best_dice:
            best_dice = vl['dice']
            torch.save(model.state_dict(), best_path)
            save_ckpt(epoch, model, optimizer, scheduler, history, best_dice, ckpt_path)
            tag = ' BEST'

        print(f'[{model_name}] Ep {epoch:02d}/{epochs} '
              f'| Loss {tr["loss"]:.4f}/{vl["loss"]:.4f} '
              f'| Dice {tr["dice"]:.4f}/{vl["dice"]:.4f} '
              f'| Acc {tr["acc"]:.4f}/{vl["acc"]:.4f}{tag}')

    elapsed = (time.time()-t0)/60

    # ── Courbes d'apprentissage → dossier du modèle ───────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history['train_loss'], label='Train', color='royalblue')
    axes[0].plot(history['val_loss'],   label='Val',   color='tomato')
    axes[0].set_title(f'[{model_name}] Perte'); axes[0].legend(); axes[0].grid(True,alpha=0.3)

    axes[1].plot(history['train_dice'], label='Train', color='royalblue')
    axes[1].plot(history['val_dice'],   label='Val',   color='tomato')
    axes[1].set_title(f'[{model_name}] Dice Score'); axes[1].legend(); axes[1].grid(True,alpha=0.3)

    axes[2].plot(history['train_acc'],  label='Train WARNING', color='royalblue', linestyle='--')
    axes[2].plot(history['val_acc'],    label='Val WARNING',   color='tomato',    linestyle='--')
    axes[2].set_title(f'[{model_name}] Accuracy (WARNING trompeuse)')
    axes[2].legend(); axes[2].grid(True,alpha=0.3)

    plt.suptitle(f'{model_name} — Courbes d\'apprentissage', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.show()

    # ── Métriques JSON → dossier du modèle ────────────────────────────────────
    metrics_summary = {
        'model'       : model_name,
        'best_dice_val': best_dice,
        'epochs_done' : epoch,
        'time_min'    : round(elapsed, 1),
        'final_train' : {k.replace('train_',''):history[k][-1] for k in history if k.startswith('train_')},
        'final_val'   : {k.replace('val_','')  :history[k][-1] for k in history if k.startswith('val_')},
        'history'     : history,
    }
    with open(metrics_path, 'w') as f:
        json.dump(metrics_summary, f, indent=2)

    print(f'\nOK [{model_name}] Terminé en {elapsed:.1f} min | Best Dice: {best_dice:.4f}')
    print(f'   Fichiers dans outputs/{model_name}/ :')
    for fname in os.listdir(model_dir):
        sz = os.path.getsize(os.path.join(model_dir,fname))/1024
        print(f'      {fname} ({sz:.0f} KB)')

    return history

print('OK train_model prête — sauvegarde dans le dossier du modèle !')

---
## Entraîner U-Net → `outputs/unet/`

In [ ]:
history_unet = train_model(unet, 'unet', lr=1e-4, epochs=EPOCHS)

## Entraîner SegNet → `outputs/segnet/`

In [ ]:
history_segnet = train_model(segnet, 'segnet', lr=1e-4, epochs=EPOCHS)

## Entraîner DeepLabV3 → `outputs/deeplabv3/`
*(lr réduit car backbone pré-entraîné)*

In [ ]:
history_deeplab = train_model(deeplab, 'deeplabv3', lr=5e-5, epochs=EPOCHS)

---
## Étape 7 — Évaluation sur le Test Set

In [ ]:
def evaluate_model(model, model_name):
    """Charge le meilleur modèle depuis son dossier et évalue sur le test set."""
    best_path = os.path.join(DIRS[model_name], f'best_{model_name}.pth')
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f'OK [{model_name}] Chargé depuis outputs/{model_name}/best_{model_name}.pth')
    else:
        print(f'WARNING  [{model_name}] Pas de checkpoint — utilise les poids actuels')

    model.eval()
    all_m = {k:[] for k in ['accuracy','dice','iou','precision','recall']}
    with torch.no_grad():
        for imgs, masks in tqdm(test_loader, desc=f'Test {model_name}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            m = compute_metrics(model(imgs), masks)
            for k in all_m: all_m[k].append(m[k])
    return {k: float(np.mean(v)) for k,v in all_m.items()}

# ── Évaluation des 3 modèles ──────────────────────────────────────────────────
print('Evaluation sur le jeu de TEST...\n')
results = {}
results['U-Net']      = evaluate_model(unet,    'unet')
results['SegNet']     = evaluate_model(segnet,  'segnet')
results['DeepLabV3']  = evaluate_model(deeplab, 'deeplabv3')

# ── Tableau récapitulatif ─────────────────────────────────────────────────────
print()
print('='*80)
print(f'  {"Modèle":<13} {"Accuracy":>10} {"Dice STAR":>10} {"IoU":>10} {"Précision":>10} {"Recall":>10}')
print('='*80)
for name, m in results.items():
    star = ' BEST' if m['dice'] == max(r['dice'] for r in results.values()) else ''
    print(f'  {name:<13} {m["accuracy"]:>10.4f} {m["dice"]:>10.4f} '
          f'{m["iou"]:>10.4f} {m["precision"]:>10.4f} {m["recall"]:>10.4f}{star}')
print('='*80)
print('  WARNING  Accuracy gonflée par déséquilibre — Dice Score = métrique principale')

# ── Sauvegarde dans outputs/comparaison/ ─────────────────────────────────────
with open(os.path.join(DIRS['comparaison'], 'resultats_finaux.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('\n Save outputs/comparaison/resultats_finaux.json')

## Étape 8 — Graphiques de comparaison → `outputs/comparaison/`

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
colors = {'U-Net':'#0E7C7B', 'SegNet':'#F26419', 'DeepLabV3':'#3B82F6'}
styles = {'U-Net':'-',       'SegNet':'--',       'DeepLabV3':'-.'}
histories = {'U-Net':history_unet, 'SegNet':history_segnet, 'DeepLabV3':history_deeplab}

# 1. Loss
for name, h in histories.items():
    ep = range(1, len(h['val_loss'])+1)
    axes[0,0].plot(ep, h['train_loss'], styles[name], color=colors[name], alpha=0.3, lw=1.5)
    axes[0,0].plot(ep, h['val_loss'],   styles[name], color=colors[name], lw=2.5, label=name)
axes[0,0].set_title('Perte — Train (pâle) / Val (solide)', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# 2. Dice
for name, h in histories.items():
    ep = range(1, len(h['val_dice'])+1)
    axes[0,1].plot(ep, h['train_dice'], styles[name], color=colors[name], alpha=0.3, lw=1.5)
    axes[0,1].plot(ep, h['val_dice'],   styles[name], color=colors[name], lw=2.5, label=name)
axes[0,1].set_title('Dice Score — Train / Val', fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# 3. Accuracy
for name, h in histories.items():
    ep = range(1, len(h['val_acc'])+1)
    axes[0,2].plot(ep, h['val_acc'], styles[name], color=colors[name], lw=2.5, label=name)
axes[0,2].set_title('Accuracy Val — WARNING Gonflée par déséquilibre', fontweight='bold')
axes[0,2].set_xlabel('Epoch'); axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

# 4. Toutes métriques — barres
metric_keys   = ['accuracy','dice','iou','precision','recall']
metric_labels = ['Accuracy\nWARNING','Dice\nSTAR','IoU','Précision','Recall']
x = np.arange(len(metric_keys)); w = 0.25
for i,(name,m) in enumerate(results.items()):
    vals = [m[k] for k in metric_keys]
    bars = axes[1,0].bar(x+i*w, vals, w, label=name,
                         color=list(colors.values())[i], alpha=0.85, edgecolor='white')
    for bar,v in zip(bars,vals):
        axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                       f'{v:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
axes[1,0].set_title('Toutes métriques — Test Set', fontweight='bold')
axes[1,0].set_xticks(x+w); axes[1,0].set_xticklabels(metric_labels)
axes[1,0].set_ylim(0,1.18); axes[1,0].legend()
axes[1,0].axhline(0.8, color='gray', lw=1, linestyle='--', alpha=0.5)
axes[1,0].grid(axis='y', alpha=0.3)

# 5. Radar
ax_r = plt.subplot(2,3,5,polar=True)
rkeys = ['dice','iou','precision','recall','accuracy']
rlbls = ['Dice','IoU','Précision','Recall','Accuracy']
N = len(rkeys)
angles = np.linspace(0,2*np.pi,N,endpoint=False).tolist()+[0]
ax_r.set_theta_offset(np.pi/2); ax_r.set_theta_direction(-1)
ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(rlbls, size=9)
ax_r.set_ylim(0.5,1.0); ax_r.grid(True,alpha=0.3)
for name,m in results.items():
    vals = [m[k] for k in rkeys]+[m[rkeys[0]]]
    ax_r.plot(angles, vals, styles[name], color=colors[name], lw=2.5, label=name)
    ax_r.fill(angles, vals, color=colors[name], alpha=0.07)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.35,1.15), fontsize=9)
ax_r.set_title('Radar — Profil des modèles', fontweight='bold', pad=20)

# 6. Résumé texte
axes[1,2].axis('off')
best_model = max(results, key=lambda k: results[k]['dice'])
txt = 'REPORT RÉSUMÉ FINAL\n\n'
for name, m in results.items():
    flag = ' STAR MEILLEUR' if name == best_model else ''
    txt += f'{name}{flag}\n'
    txt += f'  Accuracy  : {m["accuracy"]:.4f} WARNING\n'
    txt += f'  Dice      : {m["dice"]:.4f} STAR\n'
    txt += f'  IoU       : {m["iou"]:.4f}\n'
    txt += f'  Précision : {m["precision"]:.4f}\n'
    txt += f'  Recall    : {m["recall"]:.4f}\n\n'
axes[1,2].text(0.05,0.95, txt, transform=axes[1,2].transAxes,
               fontsize=10, va='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='#f8fafc', alpha=0.8))

plt.suptitle('Comparaison U-Net vs SegNet vs DeepLabV3 — ISIC', fontsize=15, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(DIRS['comparaison'], 'comparaison_complete.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Save outputs/comparaison/comparaison_complete.png')

## Étape 9 — Comparaison visuelle

In [ ]:
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

imgs_b, masks_b = next(iter(test_loader))
preds_dict = {}
model_map = [('U-Net','unet',unet),('SegNet','segnet',segnet),('DeepLabV3','deeplabv3',deeplab)]

for label, fname, model in model_map:
    best_path = os.path.join(DIRS[fname], f'best_{fname}.pth')
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(model(imgs_b.to(device))) > THRESHOLD).float().cpu()
    preds_dict[label] = pred

N = min(4, len(imgs_b))
fig, axes = plt.subplots(2+len(preds_dict), N, figsize=(4*N, 4.5*(2+len(preds_dict))))
row_labels = ['FRAME_WITH_PICTURE Image','OK Masque réel','BLUE_CIRCLE U-Net','ORANGE_CIRCLE SegNet','GREEN_CIRCLE DeepLabV3']

for col in range(N):
    img_s  = (imgs_b[col]*STD+MEAN).clamp(0,1).permute(1,2,0).numpy()
    mask_s = masks_b[col,0].numpy()
    axes[0,col].imshow(img_s);             axes[0,col].set_title(f'Image {col+1}'); axes[0,col].axis('off')
    axes[1,col].imshow(mask_s,cmap='gray');axes[1,col].axis('off')
    for ri,(name,preds) in enumerate(preds_dict.items()):
        pred_s   = preds[col,0].numpy()
        img_uint = (img_s*255).astype(np.uint8)
        ov = img_uint.copy(); ov[pred_s>0]=[255,80,80]
        blended  = cv2.addWeighted(img_uint,0.55,ov,0.45,0)
        p=pred_s.flatten().astype(float); t=mask_s.flatten().astype(float)
        tp_i=(p*t).sum(); fp_i=(p*(1-t)).sum(); fn_i=((1-p)*t).sum()
        d = (2*tp_i+1e-6)/(2*tp_i+fp_i+fn_i+1e-6)
        axes[ri+2,col].imshow(blended)
        axes[ri+2,col].set_title(f'Dice={d:.3f}',fontsize=9); axes[ri+2,col].axis('off')

for ri,lbl in enumerate(row_labels):
    axes[ri,0].set_ylabel(lbl, fontsize=11, fontweight='bold',
                           rotation=0, ha='right', va='center', labelpad=130)

plt.suptitle('Comparaison Visuelle — U-Net vs SegNet vs DeepLabV3', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(DIRS['comparaison'], 'comparaison_visuelle.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Save outputs/comparaison/comparaison_visuelle.png')

## Étape 10 — Structure finale & Conclusion

In [ ]:
best_model = max(results, key=lambda k: results[k]['dice'])

print('='*65)
print('STRUCTURE FINALE SUR GOOGLE DRIVE')
print('='*65)
print('ISIC_Project/outputs/')
for folder, path in DIRS.items():
    print(f'├── {folder}/')
    if os.path.exists(path):
        for fname in sorted(os.listdir(path)):
            sz = os.path.getsize(os.path.join(path,fname))/1024
            print(f'│   ├── {fname} ({sz:.0f} KB)')
print()
print('='*65)
print('TROPHY  RÉSULTATS FINAUX')
print('='*65)
for name, m in results.items():
    star = ' MEILLEUR' if name == best_model else ''
    print(f'  {name}{star}')
    print(f'    Dice      : {m["dice"]:.4f}')
    print(f'    IoU       : {m["iou"]:.4f}')
    print(f'    Précision : {m["precision"]:.4f}')
    print(f'    Recall    : {m["recall"]:.4f}')
    print(f'    Accuracy  : {m["accuracy"]:.4f} WARNING')
    print()
print(f'  Recommandation → {best_model}')
print('='*65)

# Export conclusion
conclusion = {
    'best_model': best_model,
    'results'   : results,
    'folders'   : {k: str(v) for k,v in DIRS.items()},
    'dataset'   : {'total':len(all_images),'train':len(train_imgs),
                   'val':len(val_imgs),'test':len(test_imgs)},
    'config'    : {'epochs':EPOCHS,'batch_size':BATCH_SIZE,
                   'img_size':IMG_SIZE,'threshold':THRESHOLD}
}
with open(os.path.join(DIRS['comparaison'],'conclusion_finale.json'),'w') as f:
    json.dump(conclusion, f, indent=2, default=float)
print('Save outputs/comparaison/conclusion_finale.json')